#### 📌 SQL Logical Execution Order

SQL clauses are **logically** evaluated in the order below — this is **not** the order you write them in a query, but the order the database engine processes them:

| Order | Clause        | Purpose                                                                 |
|-------|---------------|-------------------------------------------------------------------------|
| 1️⃣   | **FROM**      | Identify the source table(s); **JOINs** are resolved here (including `ON` conditions)  |
| 2️⃣   | **WHERE**     | Filter rows based on conditions                                          |
| 3️⃣   | **GROUP BY**  | Group filtered rows by one or more columns                              |
| 4️⃣   | **HAVING**    | Filter groups based on aggregate conditions                              |
| 5️⃣   | **SELECT**    | Choose which columns/expressions to return                              |
| 6️⃣   | **ORDER BY**  | Sort the final result set                                               |
| 7️⃣   | **LIMIT**     | Restrict the number of rows returned                                     |

> **Why this matters:**
> - You **cannot** reference a column alias defined in `SELECT` inside `WHERE` or `HAVING` because those clauses are evaluated **before** `SELECT`.
> - `HAVING` is used for filtering **after** aggregation, while `WHERE` filters **before** aggregation.
> - Window functions (e.g., `RANK()`, `DENSE_RANK()`) are evaluated during `SELECT`, so they can only appear in `SELECT` and `ORDER BY`.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkDataframe").getOrCreate()

In [0]:
columns = ["employee_id", "employee_name", "department", "salary", "manager_id"]

data = [
    (101, "John Smith",      "IT",      135000, 201),
    (102, "Alice Johnson",   "IT",      85000, 201),
    (103, "Bob Williams",    "HR",      70000, 202),
    (104, "Carol Brown",     "HR",      75000, 202),
    (105, "David Miller",    "Finance", 90000, 203),
    (106, "Emma Davis",      "Finance", 80000, 203),
    (107, "Frank Wilson",    "IT",      78000, 201),
    (108, "Grace Taylor",    "Sales",   120000, 204),
    (109, "Henry Moore",     "Sales",   68000, 204),
    (110, "Ivy Anderson",    "Marketing",76000, 205),
    (201, "Michael Scott",   "IT",     130000, 301),
    (202, "Sarah Connor",    "HR",     120000, 301),
    (203, "Robert King",     "Finance",145000, 301),
    (204, "Nancy White",     "Sales",  118000, 301),
    (205, "James Clark",     "Marketing",122000, 301),
    (301, "Dilip M G",     "CEO",140000, None),
]

df= spark.createDataFrame(data,columns)
df.createOrReplaceTempView('employees')


##### Display all employees from Sales department ordered by salary from highest to lowest. 

In [0]:
%sql
select
  *
from
  employees
where
  department = 'Sales'
order by
  salary desc

employee_id,employee_name,department,salary,manager_id
108,Grace Taylor,Sales,120000,204
204,Nancy White,Sales,118000,301
109,Henry Moore,Sales,68000,204



##### Display count of employees in each department

In [0]:
%sql
select
  department,
  count(*) as dept_count
from
  employees
group by
  department
order by
  department;

department,dept_count
CEO,1
Finance,3
HR,3
IT,4
Marketing,2
Sales,3



##### Display 2nd hightest salary record

In [0]:
%sql
select max(salary) from employees
where salary < (select max(salary) from employees);


max(salary)
140000



##### Display hightest salary record from each department

In [0]:
%sql
-- SQL solution
select
  department,
  max(salary) as highest_salary
from
  employees
group by
  department
order by department

department,highest_salary
CEO,140000
Finance,145000
HR,120000
IT,135000
Marketing,122000
Sales,120000


In [0]:
## Using Pyspark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy('department').orderBy(F.col("salary").desc())

result = (
    df.withColumn("rank", F.dense_rank().over(window_spec))
      .filter(F.col("rank") == 1)
      .select('department',"salary")
      .orderBy("department")
)

result.show()

+----------+------+
|department|salary|
+----------+------+
|       CEO|140000|
|   Finance|145000|
|        HR|120000|
|        IT|135000|
| Marketing|122000|
|     Sales|120000|
+----------+------+




##### Display 2nd hightest salary record from each department

In [0]:
%sql
with employee_ctas
as 
(select
  employee_id,
  employee_name,
  salary,
  department,
  manager_id,
  dense_rank() over (partition by department order by salary desc) as salary_rank
from
  employees)

  select `salary`, `department`, `manager_id`, `employee_id`, `employee_name` from employee_ctas
  where salary_rank = 2
  order by department

salary,department,manager_id,employee_id,employee_name
90000,Finance,203,105,David Miller
75000,HR,202,104,Carol Brown
130000,IT,301,201,Michael Scott
76000,Marketing,205,110,Ivy Anderson
118000,Sales,301,204,Nancy White



##### Find employeees earning more than company average salary

In [0]:
%sql
select *
from
  ( select *, avg(salary)  over () as avg_salary
    from employees )
where salary > avg_salary;

employee_id,employee_name,department,salary,manager_id,avg_salary
101,John Smith,IT,135000,201,103250.0
108,Grace Taylor,Sales,120000,204,103250.0
201,Michael Scott,IT,130000,301,103250.0
202,Sarah Connor,HR,120000,301,103250.0
203,Robert King,Finance,145000,301,103250.0
204,Nancy White,Sales,118000,301,103250.0
205,James Clark,Marketing,122000,301,103250.0
301,Dilip M G,CEO,140000,null,103250.0



##### Display employee who is getting more salary than his manager

In [0]:
%sql
select e.employee_name, 
       e.salary as employee_salary,
       m.employee_name as manager_name,
       m.salary as manager_salary
       from employees e 
       join employees m 
       on e.manager_id = m.employee_id
       where e.salary > m.salary;


employee_name,employee_salary,manager_name,manager_salary
John Smith,135000,Michael Scott,130000
Grace Taylor,120000,Nancy White,118000
Robert King,145000,Dilip M G,140000


In [0]:
customer_columns = ["customer_id", "customer_name", "city"]

customer_data = [
    (101, "John", "Bangalore"),
    (102, "Alice", "Mumbai"),
    (103, "Robert", "Chennai"),
    (104, "Emma", "Hyderabad"),
    (105, "David", "Pune"),
    (106, "Sophia", "Delhi"),
    (107, "James", "Bangalore")
]

customer_df = spark.createDataFrame(customer_data,customer_columns)
customer_df.createOrReplaceTempView('customers')

order_columns = ["order_id", "customer_id", "amount", "status"]

order_data = [
    (1001, 101, 500, "Delivered"),
    (1002, 102, 750, "Delivered"),
    (1003, 101, 300, "Pending"),
    (1004, 103, 1200, "Delivered"),
    (1005, 104, 450, "Cancelled"),
    (1006, 102, 900, "Delivered"),
    (1007, 104, 650, "Pending"),
    (1008, 103, 400, "Delivered"),
    (1009, 104, 1100, "Delivered"),
    (1010, 101, 800, "Delivered")
]

orders_df=spark.createDataFrame(order_data,order_columns)
orders_df.createOrReplaceTempView('orders')



##### Display each order with customer name

In [0]:
%sql
select
  order_id,
  customer_name,
  amount,
  status,
  city
from
  customers c
    join orders o
      on c.customer_id = o.customer_id

order_id,customer_name,amount,status,city
1001,John,500,Delivered,Bangalore
1002,Alice,750,Delivered,Mumbai
1003,John,300,Pending,Bangalore
1004,Robert,1200,Delivered,Chennai
1005,Emma,450,Cancelled,Hyderabad
1006,Alice,900,Delivered,Mumbai
1007,Emma,650,Pending,Hyderabad
1008,Robert,400,Delivered,Chennai
1009,Emma,1100,Delivered,Hyderabad
1010,John,800,Delivered,Bangalore



##### Display customers who has never placed an order


In [0]:
%sql
select
  customer_name,
  city
from
  customers c
    left join orders o
      on c.customer_id = o.customer_id
where order_id is null

customer_name,city
David,Pune
Sophia,Delhi
James,Bangalore



##### Display customers who has placed more than two orders

In [0]:
%sql
select
  customer_name,
  city,
  count(*) as no_of_orders
from
  customers c
     join orders o
      on c.customer_id = o.customer_id
group by customer_name, city
having count(*)  > 2;


customer_name,city,no_of_orders
John,Bangalore,3
Emma,Hyderabad,3



##### find highest spending customer based on delivery status

In [0]:
%sql
select
  customer_name,
  sum(amount) as total_spent
from
  customers c
     join orders o
      on c.customer_id = o.customer_id
where status = 'Delivered'
group by customer_name
order by total_spent desc
limit 1

customer_name,total_spent
Alice,1650


In [0]:
%sql
WITH delivered_stats
as
(select
  customer_name,
  sum(amount) as total_spent
from
  customers c
     join orders o
      on c.customer_id = o.customer_id
where status = 'Delivered'
group by customer_name)

select * from delivered_stats
where total_spent = (Select max(total_spent) from delivered_stats)

customer_name,total_spent
Alice,1650



### Find the stats of the team from the data. Finding no of matches won, lost and played

In [0]:
data = [('India','SL','India'),('SL','Aus','Aus'),('SA','Eng','Eng'),('Eng','NZ','NZ'),('Aus','India','India')]
columns = ['Team_1','Team_2','Winner']

df=spark.createDataFrame(data,columns)
df.createOrReplaceTempView("matches")

In [0]:
%sql
select * from matches;

Team_1,Team_2,Winner
India,SL,India
SL,Aus,Aus
SA,Eng,Eng
Eng,NZ,NZ
Aus,India,India


In [0]:
%sql
select team_name, count(1) as no_of_matches_played, sum(win_flag) as no_of_matches_won, count(1) - sum(win_flag) as no_of_losses
from
(select team_1 as team_name, case when team_1 = winner then 1 else 0 end as win_flag
from matches
union all 
select team_2  as team_name, case when team_2 = winner then 1 else 0 end as win_flag
from matches) A
group by team_name
order by no_of_matches_won desc


team_name,no_of_matches_played,no_of_matches_won,no_of_losses
India,2,2,0
Eng,2,1,1
Aus,2,1,1
NZ,1,1,0
SL,2,0,2
SA,1,0,1


#### Identify the number of existing customers and new customers for a given day 


In [0]:
data = [(1001, "C001", "2024-01-01", 250),
    (1002, "C002", "2024-01-01", 120),
    (1003, "C001", "2024-01-01", 450),
    (1004, "C003", "2024-01-02", 300),
    (1005, "C004", "2024-01-02", 150),
    (1006, "C002", "2024-01-02", 500),
    (1007, "C005", "2024-01-03", 75),
    (1008, "C003", "2024-01-03", 220),
    (1009, "C001", "2024-01-03", 800),
    (1010, "C004", "2024-01-03", 125)]

columns = ["order_id", "customer_id", "order_date", "order_amount"]

df=spark.createDataFrame(data,columns)
df.createOrReplaceTempView("orders")                           

In [0]:
%sql
select * from orders

order_id,customer_id,order_date,order_amount
1001,C001,2024-01-01,250
1002,C002,2024-01-01,120
1003,C001,2024-01-01,450
1004,C003,2024-01-02,300
1005,C004,2024-01-02,150
1006,C002,2024-01-02,500
1007,C005,2024-01-03,75
1008,C003,2024-01-03,220
1009,C001,2024-01-03,800
1010,C004,2024-01-03,125


###### Adding First_visit and Repeat_visit flags

In [0]:
%sql
with 
first_visit as (
  select customer_id, min(order_date) as first_visit_date
  from orders
  group by customer_id)

select o.order_date, o.customer_id, f.first_visit_date,
  case when o.order_date = f.first_visit_date then 1 else 0 end as first_visit_flag,
  case when o.order_date != f.first_visit_date then 1 else 0 end as repeat_visit_flag
from orders o
inner join first_visit f
      on o.customer_id = f.customer_id
order by  order_id

order_date,customer_id,first_visit_date,first_visit_flag,repeat_visit_flag
2024-01-01,C001,2024-01-01,1,0
2024-01-01,C002,2024-01-01,1,0
2024-01-01,C001,2024-01-01,1,0
2024-01-02,C003,2024-01-02,1,0
2024-01-02,C004,2024-01-02,1,0
2024-01-02,C002,2024-01-01,0,1
2024-01-03,C005,2024-01-03,1,0
2024-01-03,C003,2024-01-02,0,1
2024-01-03,C001,2024-01-01,0,1
2024-01-03,C004,2024-01-02,0,1


###### Inner joining one CTAS table to get the desired output

In [0]:
%sql
with first_visit as (
  select customer_id, min(order_date) as first_visit_date
  from orders
  group by customer_id)

select o.order_date, 
  sum(case when o.order_date = f.first_visit_date then 1 else 0 end) as first_visit_flag,
  sum(case when o.order_date != f.first_visit_date then 1 else 0 end) as repeat_visit_flag
from orders o
inner join first_visit f
      on o.customer_id = f.customer_id
group by o.order_date
order by o.order_date

order_date,first_visit_flag,repeat_visit_flag
2024-01-01,3,0
2024-01-02,2,1
2024-01-03,1,3


###### Creating 2 CTAS tables to get the desired output

In [0]:
%sql
with first_visit as (
  select
    customer_id,
    min(order_date) as first_visit_date
  from
    orders
  group by
    customer_id
),
visit_data as (
  select
    o.order_date,
    o.customer_id,
    f.first_visit_date,
    case
      when o.order_date = f.first_visit_date then 1
      else 0
    end as first_visit_flag,
    case
      when o.order_date != f.first_visit_date then 1
      else 0
    end as repeat_visit_flag
  from
    orders o
      inner join first_visit f
        on o.customer_id = f.customer_id
  order by
    order_id
)
select
  order_date,
  sum(first_visit_flag) as first_visits,
  sum(repeat_visit_flag) as repeat_visits
from
  visit_data
group by order_date
order by order_date

order_date,first_visits,repeat_visits
2024-01-01,3,0
2024-01-02,2,1
2024-01-03,1,3


###### using inner query with MIN() and OVER() to get the desired output

In [0]:
%sql
SELECT
    order_dt AS order_date,
    CASE WHEN order_dt = first_visit_dt THEN customer_id END AS first_visits,
    CASE WHEN order_dt > first_visit_dt THEN customer_id END AS repeat_visits
FROM (
    SELECT
        customer_id, order_date as order_dt, 
        MIN(order_date) OVER (PARTITION BY customer_id) AS first_visit_dt
    FROM orders) t
ORDER BY order_dt;

order_date,first_visits,repeat_visits
2024-01-01,C001,null
2024-01-01,C002,null
2024-01-01,C001,null
2024-01-02,null,C002
2024-01-02,C004,null
2024-01-02,C003,null
2024-01-03,null,C001
2024-01-03,null,C004
2024-01-03,null,C003
2024-01-03,C005,null


In [0]:
%sql
SELECT
    order_dt AS order_date,
    COUNT( CASE WHEN order_dt = first_visit_dt THEN customer_id END) AS first_visits,
    COUNT( CASE WHEN order_dt > first_visit_dt THEN customer_id END) AS repeat_visits
FROM (
    SELECT
        customer_id, order_date as order_dt, 
        MIN(order_date) OVER (PARTITION BY customer_id) AS first_visit_dt
    FROM orders) t
GROUP BY order_dt
ORDER BY order_dt;

order_date,first_visits,repeat_visits
2024-01-01,3,0
2024-01-02,2,1
2024-01-03,1,3


#### Enhancing with revenue from first_visit and repeat visit customers

In [0]:
%sql
SELECT * FROM ORDers limit 5

order_id,customer_id,order_date,order_amount
1001,C001,2024-01-01,250
1002,C002,2024-01-01,120
1003,C001,2024-01-01,450
1004,C003,2024-01-02,300
1005,C004,2024-01-02,150


In [0]:
%sql
SELECT
    order_dt AS order_date,
    CASE WHEN order_dt = first_visit_dt THEN customer_id END AS first_visits,
    CASE WHEN order_dt > first_visit_dt THEN customer_id END AS repeat_visits,
    CASE WHEN order_dt = first_visit_dt THEN order_amount ELSE 0 END AS first_visit_revenue,
    CASE WHEN order_dt > first_visit_dt THEN order_amount ELSE 0 END AS repeat_visit_revenue
FROM (
    SELECT
        customer_id, order_amount, order_date as order_dt, 
        MIN(order_date) OVER (PARTITION BY customer_id) AS first_visit_dt
    FROM orders) t
ORDER BY order_dt;

order_date,first_visits,repeat_visits,first_visit_revenue,repeat_visit_revenue
2024-01-01,C001,null,450,0
2024-01-01,C002,null,120,0
2024-01-01,C001,null,250,0
2024-01-02,null,C002,0,500
2024-01-02,C004,null,150,0
2024-01-02,C003,null,300,0
2024-01-03,null,C001,0,800
2024-01-03,null,C004,0,125
2024-01-03,null,C003,0,220
2024-01-03,C005,null,75,0


In [0]:
%sql
SELECT
    order_dt AS order_date,
    COUNT( CASE WHEN order_dt = first_visit_dt THEN customer_id END) AS first_visits,
    COUNT( CASE WHEN order_dt > first_visit_dt THEN customer_id END) AS repeat_visits,
    SUM(CASE WHEN order_dt = first_visit_dt THEN order_amount ELSE 0 END) AS first_visit_revenue,
    SUM(CASE WHEN order_dt > first_visit_dt THEN order_amount ELSE 0 END) AS repeat_visit_revenue
FROM (
    SELECT
        customer_id, order_amount, order_date as order_dt, 
        MIN(order_date) OVER (PARTITION BY customer_id) AS first_visit_dt
    FROM orders) t
GROUP BY order_dt
ORDER BY order_dt;

order_date,first_visits,repeat_visits,first_visit_revenue,repeat_visit_revenue
2024-01-01,3,0,820,0
2024-01-02,2,1,450,500
2024-01-03,1,3,75,1145


#### Extract the Summary of Inventory details.

In [0]:
columns = [
    "id",
    "onhandquantity",
    "onhandquantityDelta",
    "event_type",
    "event_datetime"
]

data = [
    (101, 100,  100, "Inbound",  "2026-10-13 09:00"),
    (102,  80,  20, "Outbound", "2026-09-14 11:30"),
    (103, 130,   50, "Inbound",  "2026-09-25 10:15"),
    (104, 100,  30, "Outbound", "2026-08-13 14:20"),
    (105, 200,  200, "Inbound",  "2026-07-15 08:30"),
    (106, 150,  50, "Outbound", "2026-05-12 12:00"),
    (107, 250,  100, "Inbound",  "2026-06-23 09:45"),
    (108, 210,  40, "Outbound", "2026-06-23 16:30"),
    (109, 150,  150, "Inbound",  "2026-04-16 10:00"),
    (110, 120,  30, "Outbound", "2026-02-12 13:00"),
    (111,  70,  50, "Outbound", "2026-03-13 15:00"),
    (112, 170,  100, "Inbound",  "2026-01-24 11:00")
]

#### Extract the report of employees only when there is salary changes for consecutive years

In [0]:
data = [
    (101, 2022, 50000),
    (101, 2023, 50000),
    (101, 2024, 55000),
    (102, 2022, 60000),
    (102, 2023, 65000),
    (102, 2024, 65000),
    (103, 2022, 45000),
    (103, 2023, 48000),
    (103, 2024, 52000),
    (104, 2022, 70000),
    (104, 2023, 70000),
    (104, 2024, 75000)
]
columns = ["id", "year", "salary"]

salary_df = spark.createDataFrame(data,columns)
salary_df.createOrReplaceTempView('salary')

In [0]:
%sql
with salary_data as (
  select
    id,
    year,
    salary,
    lag(salary) over (partition by id order by year) as last_yr_salary
  from
    salary
)
select
  *,
  salary - last_yr_salary as salary_delta
from
  salary_data
where
  salary <> last_yr_salary

id,year,salary,last_yr_salary,salary_delta
101,2024,55000,50000,5000
102,2023,65000,60000,5000
103,2023,48000,45000,3000
103,2024,52000,48000,4000
104,2024,75000,70000,5000


In [0]:
people = [("Alice", "Female"), ("Bob", "Male"), ("Charlie", "Male"), ("Diana", "Female"),  ("Ethan", "Male"),  ("Fiona", "Female"), ("George", "Male"), ("Hannah", "Female"),  ("Ian", "Male"), ("Julia", "Female")]

people_df = spark.createDataFrame(people, ["name", "gender"])
people_df.createOrReplaceTempView('people')

#### Write a single update query to chnage the gender, Male to Female and Female to Male

In [0]:
### Using Pyspark

from pyspark.sql.functions import when, col

print('Before Update:')
people_df.show(5)
people_df = people_df.withColumn('gender',
                                 when (col('gender')== 'Male', 'Female').
                                 when (col('gender')=='Female', 'Male'))
print('After Update:')
people_df.show(5)


Before Update:
+-------+------+
|   name|gender|
+-------+------+
|  Alice|Female|
|    Bob|  Male|
|Charlie|  Male|
|  Diana|Female|
|  Ethan|  Male|
+-------+------+
only showing top 5 rows
After Update:
+-------+------+
|   name|gender|
+-------+------+
|  Alice|  Male|
|    Bob|Female|
|Charlie|Female|
|  Diana|  Male|
|  Ethan|Female|
+-------+------+
only showing top 5 rows


In [0]:
%sql
-- Using SQL
-- update people
-- set gender=
-- CASE
--     WHEN gender= 'Male' then 'Female'
--     WHEN gender= 'Female' then 'Male'
--     ELSE gender
-- END;

### Challenge: Find Each Customer's Best Spending Streak

#### Datasets

- **customers** — contains customer information.
- **orders** — contains order amounts and dates.
- **order_status** — contains multiple status updates for each order.

#### Business Rules

1. Determine the latest status of every order based on `status_timestamp`.
2. Consider only orders whose latest status is **Delivered**.
3. Calculate each customer's delivered order amount for each calendar month.
4. A month is a **qualifying month** when the monthly amount is **at least 500**.
5. Consecutive qualifying months form a **spending streak**.
6. Missing months and months with spending below 500 break the streak.
7. Return the **best streak** for every customer.

#### Tie-Breaker Rules

If a customer has multiple streaks:

1. Select the streak with the **longest duration**.
2. If durations are equal, select the streak with the **highest total amount**.
3. If still tied, select the **earliest streak**.

###### Customers Data

In [0]:
customer_columns = [
    "customer_id",
    "customer_name"
]

customer_data = [
    ("C001", "Alice"),
    ("C002", "Bob"),
    ("C003", "Charlie"),
    ("C004", "David")
]

customers_df = spark.createDataFrame(
    customer_data,
    customer_columns
)
customers_df.createOrReplaceTempView('customers')

##### Orders Data

In [0]:
order_columns = [
    "order_id",
    "customer_id",
    "order_date",
    "order_amount"
]

order_data = [
    # Alice
    ("O1001", "C001", "2025-01-05", 600.00),
    ("O1002", "C001", "2025-02-10", 700.00),
    ("O1003", "C001", "2025-03-15", 800.00),
    ("O1004", "C001", "2025-04-20", 400.00),
    ("O1005", "C001", "2025-05-12", 900.00),

    # Bob
    ("O2001", "C002", "2025-01-08", 550.00),
    ("O2002", "C002", "2025-02-11", 600.00),
    ("O2003", "C002", "2025-03-18", 650.00),
    ("O2004", "C002", "2025-04-22", 700.00),
    ("O2005", "C002", "2025-05-25", 750.00),

    # Charlie
    ("O3001", "C003", "2025-01-03", 520.00),
    ("O3002", "C003", "2025-02-07", 530.00),
    ("O3003", "C003", "2025-03-17", 540.00),
    ("O3004", "C003", "2025-04-28", 550.00),

    # David
    ("O4001", "C004", "2025-01-14", 900.00),
    ("O4002", "C004", "2025-02-16", 300.00),
    ("O4003", "C004", "2025-03-19", 900.00),
    ("O4004", "C004", "2025-04-21", 200.00)
]

from pyspark.sql import functions as F
orders_df = (
    spark.createDataFrame(order_data, order_columns)
         .withColumn(
             "order_date",
             F.to_date("order_date", "yyyy-MM-dd")
         )
)

orders_df.createOrReplaceTempView('orders')

##### Order Status Data

In [0]:
status_columns = [
    "order_id",
    "order_status",
    "status_timestamp"
]

status_data = [
    # Alice
    ("O1001", "Placed",    "2025-01-05 09:00:00"),
    ("O1001", "Delivered", "2025-01-08 14:00:00"),

    ("O1002", "Placed",    "2025-02-10 10:00:00"),
    ("O1002", "Delivered", "2025-02-14 16:00:00"),

    ("O1003", "Placed",    "2025-03-15 08:30:00"),
    ("O1003", "Delivered", "2025-03-20 12:00:00"),

    ("O1004", "Placed",    "2025-04-20 11:00:00"),
    ("O1004", "Delivered", "2025-04-25 15:00:00"),

    ("O1005", "Placed",    "2025-05-12 09:00:00"),
    ("O1005", "Cancelled", "2025-05-13 10:00:00"),

    # Bob
    ("O2001", "Placed",    "2025-01-08 09:00:00"),
    ("O2001", "Delivered", "2025-01-12 13:00:00"),

    ("O2002", "Placed",    "2025-02-11 10:00:00"),
    ("O2002", "Shipped",   "2025-02-12 15:00:00"),
    ("O2002", "Cancelled", "2025-02-13 17:00:00"),

    ("O2003", "Placed",    "2025-03-18 10:00:00"),
    ("O2003", "Delivered", "2025-03-22 11:00:00"),

    ("O2004", "Placed",    "2025-04-22 12:00:00"),
    ("O2004", "Delivered", "2025-04-26 16:00:00"),

    ("O2005", "Placed",    "2025-05-25 09:00:00"),
    ("O2005", "Delivered", "2025-05-29 14:00:00"),

    # Charlie
    ("O3001", "Delivered", "2025-01-07 10:00:00"),
    ("O3002", "Delivered", "2025-02-12 11:00:00"),
    ("O3003", "Delivered", "2025-03-21 12:00:00"),
    ("O3004", "Delivered", "2025-05-02 13:00:00"),

    # David
    ("O4001", "Delivered", "2025-01-18 10:00:00"),
    ("O4002", "Delivered", "2025-02-20 11:00:00"),
    ("O4003", "Delivered", "2025-03-23 12:00:00"),
    ("O4004", "Delivered", "2025-04-25 13:00:00")
]

order_status_df = (
    spark.createDataFrame(status_data, status_columns)
         .withColumn(
             "status_timestamp",
             F.to_timestamp("status_timestamp", "yyyy-MM-dd HH:mm:ss")
         )
)

order_status_df.createOrReplaceTempView('order_status')

In [0]:
%sql
-- customers_df 
-- orders_df
-- order_status_df

select * from order_status limit 2

order_id,order_status,status_timestamp
O1001,Placed,2025-01-05T09:00:00.000Z
O1001,Delivered,2025-01-08T14:00:00.000Z


##### Determine the latest status of every order based on status_timestamp.

In [0]:
%sql
-- SQL
with latest_status as (
  select
    order_id,
    order_status,
    status_timestamp,
    row_number() over (partition by order_id order by status_timestamp desc) as rn
  from
    order_status
)
select
  order_id,
  order_status,
  status_timestamp
from
  latest_status
where
  rn = 1


order_id,order_status,status_timestamp
O1001,Delivered,2025-01-08T14:00:00.000Z
O1002,Delivered,2025-02-14T16:00:00.000Z
O1003,Delivered,2025-03-20T12:00:00.000Z
O1004,Delivered,2025-04-25T15:00:00.000Z
O1005,Cancelled,2025-05-13T10:00:00.000Z
O2001,Delivered,2025-01-12T13:00:00.000Z
O2002,Cancelled,2025-02-13T17:00:00.000Z
O2003,Delivered,2025-03-22T11:00:00.000Z
O2004,Delivered,2025-04-26T16:00:00.000Z
O2005,Delivered,2025-05-29T14:00:00.000Z


In [0]:
## Pyspark

from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = Window.partitionBy("order_id").orderBy(F.col("status_timestamp").desc())

order_status_df = order_status_df.withColumn("rn", F.row_number().over(window)).filter(
    F.col("rn") == 1
)
order_status_df.show(5)

+--------+------------+-------------------+---+
|order_id|order_status|   status_timestamp| rn|
+--------+------------+-------------------+---+
|   O1001|   Delivered|2025-01-08 14:00:00|  1|
|   O1002|   Delivered|2025-02-14 16:00:00|  1|
|   O1003|   Delivered|2025-03-20 12:00:00|  1|
|   O1004|   Delivered|2025-04-25 15:00:00|  1|
|   O1005|   Cancelled|2025-05-13 10:00:00|  1|
+--------+------------+-------------------+---+
only showing top 5 rows


##### Consider only orders whose latest status is Delivered.

In [0]:
window = Window.partitionBy(F.col("order_id")).orderBy(F.col("status_timestamp").desc())

order_status_df = order_status_df.withColumn("rn", F.row_number().over(window)).filter(
    (F.col("rn") == 1) & (F.col("order_status") == "Delivered"))

order_status_df.show()

+--------+------------+-------------------+---+
|order_id|order_status|   status_timestamp| rn|
+--------+------------+-------------------+---+
|   O1001|   Delivered|2025-01-08 14:00:00|  1|
|   O1002|   Delivered|2025-02-14 16:00:00|  1|
|   O1003|   Delivered|2025-03-20 12:00:00|  1|
|   O1004|   Delivered|2025-04-25 15:00:00|  1|
|   O2001|   Delivered|2025-01-12 13:00:00|  1|
|   O2003|   Delivered|2025-03-22 11:00:00|  1|
|   O2004|   Delivered|2025-04-26 16:00:00|  1|
|   O2005|   Delivered|2025-05-29 14:00:00|  1|
|   O3001|   Delivered|2025-01-07 10:00:00|  1|
|   O3002|   Delivered|2025-02-12 11:00:00|  1|
|   O3003|   Delivered|2025-03-21 12:00:00|  1|
|   O3004|   Delivered|2025-05-02 13:00:00|  1|
|   O4001|   Delivered|2025-01-18 10:00:00|  1|
|   O4002|   Delivered|2025-02-20 11:00:00|  1|
|   O4003|   Delivered|2025-03-23 12:00:00|  1|
|   O4004|   Delivered|2025-04-25 13:00:00|  1|
+--------+------------+-------------------+---+



##### Calculate each customer’s delivered order amount for each calendar month.

In [0]:
%sql
select
  o.customer_id,
  date_format(o.order_date, 'yyyy-MM') AS order_month,
  sum(o.order_amount) as month_expenditure
from
  orders o
    join order_status os
      on o.order_id = os.order_id
where
  order_status = 'Delivered'
group by
  customer_id,
  date_format(o.order_date, 'yyyy-MM')
order by
  customer_id,
  date_format(o.order_date, 'yyyy-MM')

customer_id,order_month,month_expenditure
C001,2025-01,600.0
C001,2025-02,700.0
C001,2025-03,800.0
C001,2025-04,400.0
C002,2025-01,550.0
C002,2025-03,650.0
C002,2025-04,700.0
C002,2025-05,750.0
C003,2025-01,520.0
C003,2025-02,530.0


In [0]:
result_df = (
    orders_df.alias("o")
    .join(order_status_df.alias("os"),
        F.col("o.order_id") == F.col("os.order_id"),"inner")
    .filter(F.col("order_status") == "Delivered")
    .groupBy(F.col("o.customer_id"),
            F.date_format(F.col("order_date"), "yyyy-MM").alias("order_month"))
    .agg(F.sum("order_amount").alias("month_expenditure"))
    .orderBy(F.col("customer_id"), F.col("order_month")))

result_df.show()

+-----------+-----------+-----------------+
|customer_id|order_month|month_expenditure|
+-----------+-----------+-----------------+
|       C001|    2025-01|            600.0|
|       C001|    2025-02|            700.0|
|       C001|    2025-03|            800.0|
|       C001|    2025-04|            400.0|
|       C002|    2025-01|            550.0|
|       C002|    2025-03|            650.0|
|       C002|    2025-04|            700.0|
|       C002|    2025-05|            750.0|
|       C003|    2025-01|            520.0|
|       C003|    2025-02|            530.0|
|       C003|    2025-03|            540.0|
|       C003|    2025-04|            550.0|
|       C004|    2025-01|            900.0|
|       C004|    2025-02|            300.0|
|       C004|    2025-03|            900.0|
|       C004|    2025-04|            200.0|
+-----------+-----------+-----------------+



##### A month is a qualifying month when the monthly amount is at least 500.

In [0]:
result_df = (orders_df.alias("o")
    .join(order_status_df.alias("os"), F.col("o.order_id") == F.col("os.order_id"), "inner")
    .filter(F.col("os.order_status") == "Delivered")
    .groupBy(
        F.col("o.customer_id"),
        F.date_format(F.col("o.order_date"), "yyyy-MM").alias("order_month"))
    .agg(F.sum("o.order_amount").alias("Monthly_expenditure"))
    .orderBy(F.col("customer_id"), F.col("order_month"))
    .filter(F.col("Monthly_expenditure") >= 500))

result_df.show()

+-----------+-----------+-------------------+
|customer_id|order_month|Monthly_expenditure|
+-----------+-----------+-------------------+
|       C001|    2025-01|              600.0|
|       C001|    2025-02|              700.0|
|       C001|    2025-03|              800.0|
|       C002|    2025-01|              550.0|
|       C002|    2025-03|              650.0|
|       C002|    2025-04|              700.0|
|       C002|    2025-05|              750.0|
|       C003|    2025-01|              520.0|
|       C003|    2025-02|              530.0|
|       C003|    2025-03|              540.0|
|       C003|    2025-04|              550.0|
|       C004|    2025-01|              900.0|
|       C004|    2025-03|              900.0|
+-----------+-----------+-------------------+



In [0]:
%sql
select
  o.customer_id,
  date_format(o.order_date, 'yyyy-MM') AS order_month,
  sum(o.order_amount) as month_expenditure
from
  orders o
    join order_status os
      on o.order_id = os.order_id
where
  order_status = 'Delivered'
group by
  customer_id,
  date_format(o.order_date, 'yyyy-MM')
having 
  month_expenditure >= 500
order by
  customer_id,
  date_format(o.order_date, 'yyyy-MM')

customer_id,order_month,month_expenditure
C001,2025-01,600.0
C001,2025-02,700.0
C001,2025-03,800.0
C002,2025-01,550.0
C002,2025-03,650.0
C002,2025-04,700.0
C002,2025-05,750.0
C003,2025-01,520.0
C003,2025-02,530.0
C003,2025-03,540.0


##### Consecutive qualifying months form a spending streak.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

result_df = (orders_df.alias("o")
    .join(order_status_df.alias("os"), F.col("o.order_id") == F.col("os.order_id"), "inner")
    .filter(F.col("os.order_status") == "Delivered")
    .groupBy(F.col("o.customer_id"),
             F.date_format(F.col("o.order_date"), "yyyy-MM").alias("order_month"))
    .agg(F.sum("o.order_amount").alias("Monthly_expenditure"))
    .orderBy(F.col("customer_id"), F.col("order_month"))
    .filter(F.col("Monthly_expenditure") >= 500))

window = (Window.partitionBy("customer_id").orderBy("order_month"))

# Get previous qualifying month
df = result_df.withColumn("prev_month",F.lag("order_month").over(window))
print("With Prev_month column")
df.show(5)

# Identify where a new streak starts
df = df.withColumn("new_streak",
    F.when(F.add_months(F.to_date("prev_month", "yyyy-MM"), 1) == F.to_date("order_month", "yyyy-MM"),0)
    .otherwise(1))
print("With new_streak column")
df.show(5)

# Generate streak number using cumulative sum
streak_window = (Window.partitionBy("customer_id").orderBy("order_month").rowsBetween(Window.unboundedPreceding, Window.currentRow))

df = df.withColumn("streak",F.sum("new_streak").over(streak_window))
print("With streak column")
df.show(5)

df.select("customer_id","order_month","Monthly_expenditure","streak").show(6)

With Prev_month column
+-----------+-----------+-------------------+----------+
|customer_id|order_month|Monthly_expenditure|prev_month|
+-----------+-----------+-------------------+----------+
|       C001|    2025-01|              600.0|      NULL|
|       C001|    2025-02|              700.0|   2025-01|
|       C001|    2025-03|              800.0|   2025-02|
|       C002|    2025-01|              550.0|      NULL|
|       C002|    2025-03|              650.0|   2025-01|
+-----------+-----------+-------------------+----------+
only showing top 5 rows
With new_streak column
+-----------+-----------+-------------------+----------+----------+
|customer_id|order_month|Monthly_expenditure|prev_month|new_streak|
+-----------+-----------+-------------------+----------+----------+
|       C001|    2025-01|              600.0|      NULL|         1|
|       C001|    2025-02|              700.0|   2025-01|         0|
|       C001|    2025-03|              800.0|   2025-02|         0|
|       C